# Knowledge Base Ingestion (Gemini + BM25)
This notebook ingests the corpus into a Qdrant Cloud instance using:
- **Dense Embeddings**: Google `gemini-embedding-001`
- **Sparse Vectors**: Qdrant's `BM25` (via `fastembed`)

## Prerequisites
Ensure you have a `.env` file in the root directory with the following variables:
- `QDRANT_URL`: Your Qdrant Cloud Cluster URL (e.g., `https://xyz.us-east4-0.gcp.cloud.qdrant.io`)
- `QDRANT_API_KEY`: Your Qdrant Cloud API Key
- `GOOGLE_API_KEY`: Your Google AI Studio API Key

**Target Instance**: Configured via `.env`

In [15]:
import os
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from pathlib import Path
from qdrant_client import QdrantClient, models
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from fastembed import SparseTextEmbedding

# Load environment variables
load_dotenv()

# Configuration
class Config:
    QDRANT_URL = os.getenv("QDRANT_URL")
    QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
    COLLECTION_NAME = "knowledgebase_collection"
    
    GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
    GEMINI_MODEL = "models/gemini-embedding-001"
    
    CORPUS_PATH = Path("data/corpus.parquet")

if not Config.QDRANT_API_KEY or not Config.QDRANT_URL:
    print("⚠️ WARNING: QDRANT_API_KEY or QDRANT_URL not found. Ensure they are set in your .env file.")
else:
    print("Configuration loaded.")

Configuration loaded.


In [16]:
# Initialize Client
# increased timeout to handle cloud latency
client = QdrantClient(
    url=Config.QDRANT_URL, 
    api_key=Config.QDRANT_API_KEY,
    timeout=60
)

# Initialize Gemini Embeddings
gemini_embeddings = GoogleGenerativeAIEmbeddings(
    model=Config.GEMINI_MODEL,
    google_api_key=Config.GOOGLE_API_KEY
)

print("Client and Gemini Embeddings initialized.")

Client and Gemini Embeddings initialized.


In [17]:
# Load Corpus
if Config.CORPUS_PATH.exists():
    corpus_df = pd.read_parquet(Config.CORPUS_PATH)
    print(f"Loaded {len(corpus_df)} documents from {Config.CORPUS_PATH}")
else:
    raise FileNotFoundError(f"Corpus file not found: {Config.CORPUS_PATH}")

Loaded 2469 documents from data\corpus.parquet


In [18]:
def ingest_data():
    # 1. Check Embedding Dimension dynamically
    print("Checking embedding dimension...")
    try:
        sample_vec = gemini_embeddings.embed_query("test")
        gemini_dim = len(sample_vec)
        print(f"Gemini Dimension: {gemini_dim}")
    except Exception as e:
        print(f"Error getting embedding dimension: {e}")
        return

    # 2. Recreate Collection
    print(f"Recreating collection '{Config.COLLECTION_NAME}'...")
    # recreate_collection is deprecated, using delete + create instead
    # Check if exists first to avoid error
    if client.collection_exists(collection_name=Config.COLLECTION_NAME):
        client.delete_collection(collection_name=Config.COLLECTION_NAME)
    
    client.create_collection(
        collection_name=Config.COLLECTION_NAME,
        vectors_config={
            "gemini": models.VectorParams(size=gemini_dim, distance=models.Distance.COSINE)
        },
        sparse_vectors_config={
            "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
        }
    )
    
    # 3. Create Payload Indexes
    print("Creating payload indexes...")
    # Matching schema from evaluation.ipynb
    client.create_payload_index(collection_name=Config.COLLECTION_NAME, field_name="metadata.plant", field_schema=models.PayloadSchemaType.TEXT)
    client.create_payload_index(collection_name=Config.COLLECTION_NAME, field_name="metadata.disease", field_schema=models.PayloadSchemaType.TEXT)
    client.create_payload_index(collection_name=Config.COLLECTION_NAME, field_name="metadata.type", field_schema=models.PayloadSchemaType.KEYWORD)
    client.create_payload_index(collection_name=Config.COLLECTION_NAME, field_name="metadata.source", field_schema=models.PayloadSchemaType.KEYWORD)
    
    # 4. Ingest
    points = []
    # Reduced batch size to match evaluation.ipynb and prevent WriteTimeout on Cloud
    batch_size = 25 
    docs = corpus_df.to_dict('records')
    
    print(f"Starting ingestion of {len(docs)} documents...")
    for doc in tqdm(docs):
        content = doc['contents']
        doc_id = doc['doc_id']
        metadata = doc['metadata'] if doc['metadata'] else {}
        
        # Generate Embeddings
        try:
            gemini_vec = gemini_embeddings.embed_query(content)
            
            point = models.PointStruct(
                id=doc_id,
                vector={
                    "gemini": gemini_vec,
                    # Use models.Document to let Qdrant Client handle BM25 generation (matches evaluation.ipynb)
                    "bm25": models.Document(
                        text=content,
                        model="Qdrant/bm25"
                    )
                },
                payload={
                    "page_content": content,
                    "metadata": metadata
                }
            )
            points.append(point)
        except Exception as e:
            print(f"Error processing doc {doc_id}: {e}")
            continue

        if len(points) >= batch_size:
            # Retries can also be handled by the client if configured, 
            # but usually just increasing timeout is enough.
            client.upsert(collection_name=Config.COLLECTION_NAME, points=points)
            points = []
            
    if points:
        client.upsert(collection_name=Config.COLLECTION_NAME, points=points)
        
    print("Ingestion complete!")

# Run Ingestion
ingest_data()

Checking embedding dimension...
Gemini Dimension: 3072
Recreating collection 'knowledgebase_collection'...
Creating payload indexes...
Starting ingestion of 2469 documents...


  0%|          | 0/2469 [00:00<?, ?it/s]

Ingestion complete!


In [2]:

import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from qdrant_client import models

load_dotenv()

_client = QdrantClient(url=os.getenv("QDRANT_URL"), api_key=os.getenv("QDRANT_API_KEY"), timeout=60)
_embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=os.getenv("GOOGLE_API_KEY"))
_collection = "knowledgebase_collection"

# --- Collection Info ---
info = _client.get_collection(_collection)
print(f"Collection   : {_collection}")
print(f"Points count : {info.points_count}")
print(f"Status       : {info.status}")

# --- Sample dense + sparse query ---
query = "plant disease symptoms"
query_vec = _embeddings.embed_query(query)

results = _client.query_points(
    collection_name=_collection,
    prefetch=[
        models.Prefetch(query=query_vec, using="gemini", limit=5),
        models.Prefetch(
            query=models.Document(text=query, model="Qdrant/bm25"),
            using="bm25",
            limit=5,
        ),
    ],
    query=models.FusionQuery(fusion=models.Fusion.RRF),
    limit=3,
    with_payload=True,
)

print(f"\nTop-3 results for query: '{query}'")
for i, r in enumerate(results.points, 1):
    meta = r.payload.get("metadata", {})
    snippet = r.payload.get("page_content", "")[:120].replace("\n", " ")
    print(f"  {i}. [score={r.score:.4f}] plant={meta.get('plant')} | {snippet}...")


Collection   : knowledgebase_collection
Points count : 2458
Status       : green


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

french.txt:   0%|          | 0.00/813 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

danish.txt:   0%|          | 0.00/424 [00:00<?, ?B/s]

...

spanish.txt: 0.00B [00:00, ?B/s]

russian.txt: 0.00B [00:00, ?B/s]

swedish.txt:   0%|          | 0.00/559 [00:00<?, ?B/s]

turkish.txt:   0%|          | 0.00/260 [00:00<?, ?B/s]


Top-3 results for query: 'plant disease symptoms'
  1. [score=0.5000] plant=Yams | Disease: Yam Mosaic Disease Yam Mosaic Potyvirus affecting Yams (Part 1)  Symptoms: The common symptoms are infected lea...
  2. [score=0.5000] plant=Cassava (manioc) | Disease: Cassava Mosaic Disease African Cassava Mosaic Virus (ACMV) affecting Cassava (manioc) (Part 1)  Symptoms: Disco...
  3. [score=0.4000] plant=Tomato | Disease: Verticillium Wilt affecting Tomato  Symptoms: Symptoms appear first on lower leaves and spread upwards; initial...
